# ⚽ FM Save Copilot

Welcome! This notebook turns your Football Manager 2024 squad into a Director of Football report — no installation needed.

**You'll need:**
1. Your squad exported from FM24 as an HTML file ([see the export guide](https://github.com/laweh-dev/fm24-sporting-director/blob/main/VIEW-SETUP.md))
2. *(Optional)* An Anthropic API key for the written report — costs about 2p

**How to use:** Click the ▶️ button on each cell in order, or use **Runtime → Run all**.

In [ ]:
#@title Step 1: Install the tool (click ▶️, wait ~60 seconds) { display-mode: "form" }
import os, sys

# Clone the repo so data files (roles.yaml, etc.) are available
if not os.path.exists('fm24-sporting-director'):
    !git clone -q https://github.com/laweh-dev/fm24-sporting-director.git

%cd fm24-sporting-director
!pip install -q -e . 2>&1 | tail -3

# Make fm_copilot importable directly (survives if Colab restarts after install)
_src = os.path.join(os.path.abspath('.'), 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)

print('✅ Installed! Move to the next step.')

## Step 2: Upload your FM24 exports

Run the next cell and click **Choose Files** to upload:
- Your **squad export** — save it as `squad.html` from the FM24 Squad screen
- Your **market export** — *(optional)* save as `market.html` from the Scouting screen

Don't have these yet? Follow the [export guide](https://github.com/laweh-dev/fm24-sporting-director/blob/main/VIEW-SETUP.md) first.

In [ ]:
#@title Step 2: Upload your squad files { display-mode: "form" }
from google.colab import files
import os, shutil

os.makedirs('data_uploads', exist_ok=True)
print('Upload your squad.html (and optionally market.html):')
uploaded = files.upload()

for filename in uploaded:
    dest = f'data_uploads/{filename}'
    shutil.move(filename, dest)
    print(f'  ✅ Saved to {dest}')

if 'squad.html' not in [os.path.basename(f) for f in os.listdir('data_uploads')]:
    print('⚠️  No squad.html found — make sure you name the file squad.html')
else:
    print('\nReady for Step 3!')

## Step 3: Configure your club

Fill in the form on the right, then run the cell. These details shape the analysis and the report.

In [ ]:
#@title Step 3: Configure your club { display-mode: "form" }

club_name       = 'My Club'          #@param {type:"string"}
league          = 'My League'        #@param {type:"string"}
transfer_budget = '15m'              #@param {type:"string"}
wage_budget     = '100k'             #@param {type:"string"}
formation       = '4-3-3'            #@param {type:"string"}
board_objective = 'Comfortable mid-table'  #@param ["Win the league", "Compete for Europe", "Comfortable mid-table", "Avoid relegation", "Survive / overachieve"]
pressing        = 'Mixed'            #@param ["High press", "Mixed", "Low block"]
build_up        = 'Balanced'         #@param ["Patient possession", "Balanced", "Direct"]
def_line        = 'Standard'         #@param ["High", "Standard", "Deep"]
priority_positions_str = 'LB, RB'   #@param {type:"string"}
dof_mode        = 'edwards'          #@param ["edwards", "monchi", "edu"]

# Parse comma-separated priorities
priority_positions = [p.strip().upper() for p in priority_positions_str.split(',') if p.strip()]

# Ensure fm_copilot is importable (fallback if Step 1 ran before a restart)
import sys, os as _os
_src = _os.path.join(_os.path.abspath('.'), 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)

from fm_copilot.wizard import generate_context_from_form
generate_context_from_form({
    'club_name':         club_name,
    'league':            league,
    'transfer_budget':   transfer_budget,
    'wage_budget':       wage_budget,
    'formation':         formation,
    'board_objective':   board_objective,
    'pressing':          pressing,
    'build_up':          build_up,
    'def_line':          def_line,
    'priority_positions': priority_positions,
    'dof_mode':          dof_mode,
})
print(f'\n✅ {club_name} configured in {dof_mode.title()} mode!')

## Step 4: Add your API key *(optional — skip for free mode)*

For the written Director of Football report, paste your Anthropic API key below. Get one at [console.anthropic.com](https://console.anthropic.com) — it costs about **2p per report** and £5 lasts hundreds of reports.

**Skip this** to get the free version (all the analysis, scores, and radar charts — just without the written narrative).

Your key is entered securely and is not saved in the notebook.

In [ ]:
#@title Step 4: Enter your API key (or press Enter to skip) { display-mode: "form" }
import getpass
api_key = getpass.getpass('Paste your Anthropic API key (or press Enter to skip): ')
if api_key:
    print('✅ Key received (not shown). Will generate full written report.')
else:
    print('Running in free mode — analysis and charts, no written narrative.')

## Step 5: Generate your report! 🎉

Run the next cell. It'll crunch the numbers (free, on Google's computer) and — if you entered a key — call the AI to write the narrative (~2p). Takes 15–60 seconds.

In [ ]:
#@title Step 5: Generate the report { display-mode: "form" }
import sys, os
_src = os.path.join(os.path.abspath('.'), 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)

from fm_copilot.pipeline import run_report

os.makedirs('output', exist_ok=True)

market_file = 'data_uploads/market.html'

config = {
    'squad_file':            'data_uploads/squad.html',
    'market_file':           market_file if os.path.exists(market_file) else '',
    'roles_file':            'data/roles.yaml',
    'archetypes_file':       'data/archetypes.yaml',
    'attribute_keys_file':   'data/attribute-keys.yaml',
    'context_dir':           'context',
    'dof_mode':              dof_mode,
    'api_key':               api_key if api_key else '',
    'model':                 'claude-haiku-4-5',
    'output_file':           'output/report.html',
    'candidate_threshold':   60,
    'candidates_per_position': 3,
}

report_path = run_report(config)
print(f'\n✅ Report saved to {report_path}')

In [ ]:
#@title Step 6: View and download your report { display-mode: "form" }
from IPython.display import HTML, display
from google.colab import files

print('📄 Displaying report below...')
print('   (It may look best at 100% zoom in your browser)')
print()

with open(report_path, encoding='utf-8') as f:
    report_html = f.read()

# Show inline
display(HTML(f'<div style="height:700px;overflow:auto;border:1px solid #333;border-radius:8px">{report_html}</div>'))

# Offer download
print('\nDownloading report...')
files.download(report_path)

---

## 🔄 Generating another report

To generate a new report (e.g. after a transfer window):

1. Re-upload your updated `squad.html` and `market.html` in **Step 2**
2. Update your priorities in **Step 3** if they've changed
3. Re-run **Step 5** and **Step 6**

Your club config stays saved for the session — you don't need to re-enter it unless something changes.

---

Made with ⚽ by [Michael Laweh](https://github.com/laweh-dev/fm24-sporting-director)  
*Not affiliated with Sports Interactive or SEGA.*